# Mistral AI Scoring - Progress Monitoring & Analysis

This notebook helps you:
1. **Monitor scoring progress** in real-time
2. **Check data quality** as scoring proceeds
3. **Analyze results** incrementally
4. **Visualize distributions** and relationships

## Current Scoring Status

- **Script**: `./scripts/run_scoring_mistral.sh`
- **Input**: 144,439 conversations
- **Output**: `data/scores/wildchat_full_scored_mistral.csv`
- **Checkpoints**: Every 1,000 conversations
- **Expected time**: ~50 hours with conservative delays

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Libraries imported successfully
Current time: 2025-11-03 19:51:23


## 1. Check Scoring Progress

First, let's see how many conversations have been scored so far.

In [2]:
# File paths
SCORED_FILE = '../data/scores/wildchat_full_scored_mistral.csv'
ORIGINAL_FILE = '../data/filtered/wildchat_full_preprocessed.csv'

# Total expected
TOTAL_CONVERSATIONS = 144439

# Check if scored file exists
if os.path.exists(SCORED_FILE):
    # Get file size
    file_size_mb = os.path.getsize(SCORED_FILE) / (1024 * 1024)
    
    # Count rows (subtract 1 for header)
    with open(SCORED_FILE, 'r') as f:
        scored_count = sum(1 for line in f) - 1
    
    # Calculate progress
    progress_pct = (scored_count / TOTAL_CONVERSATIONS) * 100
    remaining = TOTAL_CONVERSATIONS - scored_count
    
    print("="*70)
    print("📊 SCORING PROGRESS")
    print("="*70)
    print(f"✓ File exists: {SCORED_FILE}")
    print(f"  File size: {file_size_mb:.2f} MB")
    print(f"\n📈 Progress:")
    print(f"  Scored:     {scored_count:,} conversations")
    print(f"  Remaining:  {remaining:,} conversations")
    print(f"  Progress:   {progress_pct:.2f}%")
    print(f"\n⏱️  Estimated remaining time (at 0.8 conv/s):")
    print(f"  {remaining/(0.8*3600):.1f} hours")
    print("="*70)
    
else:
    print("❌ Scored file not found yet!")
    print(f"   Looking for: {SCORED_FILE}")
    print("\n💡 The file will be created after the first checkpoint (1000 conversations)")
    print("   Run: ./scripts/run_scoring_mistral.sh")

❌ Scored file not found yet!
   Looking for: ../data/scores/wildchat_full_scored_mistral.csv

💡 The file will be created after the first checkpoint (1000 conversations)
   Run: ./scripts/run_scoring_mistral.sh


## 2. Load and Preview Scored Data

Load the scored conversations that are available so far.

In [ ]:
# Load scored data if available
if os.path.exists(SCORED_FILE):
    print("Loading scored data...")
    df_scored = pd.read_csv(SCORED_FILE)
    
    print(f"\n✓ Loaded {len(df_scored):,} scored conversations")
    print(f"\nDataset shape: {df_scored.shape}")
    print(f"\nColumns: {list(df_scored.columns)}")
    print(f"\nFirst few rows:")
    display(df_scored.head())
    
    # Check data types
    print(f"\nData types for score columns:")
    print(f"  empathy_score: {df_scored['empathy_score'].dtype}")
    print(f"  attachment_score: {df_scored['attachment_score'].dtype}")
else:
    print("⚠️  No scored data available yet. Run the progress check cell above first.")

## 3. Data Quality Check

Check for missing scores and data quality issues.

In [ ]:
# Data quality checks
if 'df_scored' in locals():
    print("="*70)
    print("📋 DATA QUALITY CHECK")
    print("="*70)
    
    total_rows = len(df_scored)
    missing_empathy = df_scored['empathy_score'].isna().sum()
    missing_attachment = df_scored['attachment_score'].isna().sum()
    
    print(f"\n✓ Total conversations: {total_rows:,}")
    print(f"\n❌ Missing scores:")
    print(f"  Empathy:    {missing_empathy:,} ({missing_empathy/total_rows*100:.2f}%)")
    print(f"  Attachment: {missing_attachment:,} ({missing_attachment/total_rows*100:.2f}%)")
    
    # Check score ranges
    if missing_empathy < total_rows:
        valid_empathy = df_scored['empathy_score'].dropna()
        print(f"\n📊 Empathy scores:")
        print(f"  Min: {valid_empathy.min():.0f}, Max: {valid_empathy.max():.0f}")
        print(f"  Mean: {valid_empathy.mean():.2f}, Median: {valid_empathy.median():.0f}")
        
        # Check for out-of-range values
        out_of_range_emp = ((valid_empathy < 1) | (valid_empathy > 7)).sum()
        if out_of_range_emp > 0:
            print(f"  ⚠️  Out of range (not 1-7): {out_of_range_emp}")
    
    if missing_attachment < total_rows:
        valid_attachment = df_scored['attachment_score'].dropna()
        print(f"\n📊 Attachment scores:")
        print(f"  Min: {valid_attachment.min():.0f}, Max: {valid_attachment.max():.0f}")
        print(f"  Mean: {valid_attachment.mean():.2f}, Median: {valid_attachment.median():.0f}")
        
        # Check for out-of-range values
        out_of_range_att = ((valid_attachment < 1) | (valid_attachment > 7)).sum()
        if out_of_range_att > 0:
            print(f"  ⚠️  Out of range (not 1-7): {out_of_range_att}")
    
    print("="*70)
else:
    print("⚠️  Load the data first!")

## 4. Score Distributions

Visualize the distribution of empathy and attachment scores.

In [ ]:
# Plot score distributions
if 'df_scored' in locals() and len(df_scored) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Empathy scores
    empathy_data = df_scored['empathy_score'].dropna()
    if len(empathy_data) > 0:
        axes[0].hist(empathy_data, bins=np.arange(0.5, 8.5, 1), 
                     color='skyblue', edgecolor='black', alpha=0.7)
        axes[0].set_title(f'Empathy Score Distribution (n={len(empathy_data):,})', 
                         fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Empathy Score (1-7)', fontsize=12)
        axes[0].set_ylabel('Frequency', fontsize=12)
        axes[0].set_xticks(range(1, 8))
        axes[0].axvline(empathy_data.mean(), color='red', linestyle='--', 
                       label=f'Mean: {empathy_data.mean():.2f}')
        axes[0].legend()
        axes[0].grid(axis='y', alpha=0.3)
    
    # Attachment scores
    attachment_data = df_scored['attachment_score'].dropna()
    if len(attachment_data) > 0:
        axes[1].hist(attachment_data, bins=np.arange(0.5, 8.5, 1), 
                     color='lightcoral', edgecolor='black', alpha=0.7)
        axes[1].set_title(f'Attachment Score Distribution (n={len(attachment_data):,})', 
                         fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Attachment Score (1-7)', fontsize=12)
        axes[1].set_ylabel('Frequency', fontsize=12)
        axes[1].set_xticks(range(1, 8))
        axes[1].axvline(attachment_data.mean(), color='red', linestyle='--', 
                       label=f'Mean: {attachment_data.mean():.2f}')
        axes[1].legend()
        axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print value counts
    print("\nEmpathy Score Distribution:")
    print(df_scored['empathy_score'].value_counts().sort_index())
    print("\nAttachment Score Distribution:")
    print(df_scored['attachment_score'].value_counts().sort_index())
else:
    print("⚠️  Load the data first!")

## 5. Correlation Analysis

Examine the relationship between empathy and attachment scores.

In [ ]:
# Correlation analysis
if 'df_scored' in locals() and len(df_scored) > 0:
    # Calculate correlation
    df_complete = df_scored.dropna(subset=['empathy_score', 'attachment_score'])
    
    if len(df_complete) > 10:
        correlation = df_complete[['empathy_score', 'attachment_score']].corr()
        corr_coef = correlation.loc['empathy_score', 'attachment_score']
        
        print("="*70)
        print("📊 CORRELATION ANALYSIS")
        print("="*70)
        print(f"\nPearson Correlation: r = {corr_coef:.3f}")
        print(f"Sample size: n = {len(df_complete):,}")
        print("\nCorrelation Matrix:")
        print(correlation)
        print("="*70)
        
        # Scatter plot
        plt.figure(figsize=(10, 6))
        
        # Sample if too many points (for performance)
        if len(df_complete) > 10000:
            sample_size = 10000
            df_plot = df_complete.sample(n=sample_size, random_state=42)
            title_suffix = f" (showing {sample_size:,} random samples)"
        else:
            df_plot = df_complete
            title_suffix = ""
        
        plt.scatter(df_plot['empathy_score'], df_plot['attachment_score'], 
                   alpha=0.4, s=30, color='steelblue', edgecolor='none')
        plt.xlabel('Empathy Score (T)', fontsize=12)
        plt.ylabel('Attachment Score (Y)', fontsize=12)
        plt.title(f'Empathy vs Attachment{title_suffix}', fontsize=14, fontweight='bold')
        plt.xticks(range(1, 8))
        plt.yticks(range(1, 8))
        plt.grid(alpha=0.3)
        
        # Add correlation coefficient
        plt.text(0.05, 0.95, f'Pearson r = {corr_coef:.3f}\nn = {len(df_complete):,}', 
                transform=plt.gca().transAxes, fontsize=12, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Not enough complete cases for correlation analysis yet")
else:
    print("⚠️  Load the data first!")

## 6. Scores by Model

Compare empathy and attachment scores across different LLM models.

In [ ]:
# Scores by model
if 'df_scored' in locals() and len(df_scored) > 0:
    print("Empathy Scores by Model:")
    print("="*70)
    empathy_by_model = df_scored.groupby('model')['empathy_score'].agg(['mean', 'std', 'count'])
    empathy_by_model = empathy_by_model.sort_values('mean', ascending=False)
    print(empathy_by_model)
    
    print("\n" + "="*70)
    print("\nAttachment Scores by Model:")
    print("="*70)
    attachment_by_model = df_scored.groupby('model')['attachment_score'].agg(['mean', 'std', 'count'])
    attachment_by_model = attachment_by_model.sort_values('mean', ascending=False)
    print(attachment_by_model)
    
    # Box plots
    if len(df_scored) > 50:
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Empathy by model
        df_scored.boxplot(column='empathy_score', by='model', ax=axes[0])
        axes[0].set_title('Empathy Scores by Model', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Model', fontsize=12)
        axes[0].set_ylabel('Empathy Score', fontsize=12)
        axes[0].get_figure().suptitle('')
        
        # Attachment by model
        df_scored.boxplot(column='attachment_score', by='model', ax=axes[1])
        axes[1].set_title('Attachment Scores by Model', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Model', fontsize=12)
        axes[1].set_ylabel('Attachment Score', fontsize=12)
        axes[1].get_figure().suptitle('')
        
        plt.tight_layout()
        plt.show()
else:
    print("⚠️  Load the data first!")

## 7. Summary

Quick summary of current progress and key findings.

In [ ]:
# Generate summary
if 'df_scored' in locals() and len(df_scored) > 0:
    print("="*70)
    print("📊 SCORING SUMMARY")
    print("="*70)
    print(f"\n✓ Conversations scored: {len(df_scored):,} / {TOTAL_CONVERSATIONS:,}")
    print(f"  Progress: {len(df_scored)/TOTAL_CONVERSATIONS*100:.1f}%")
    
    if 'empathy_data' in locals() and len(empathy_data) > 0:
        print(f"\n📈 Empathy Scores (Treatment):")
        print(f"  Mean: {empathy_data.mean():.2f} (SD: {empathy_data.std():.2f})")
        print(f"  Median: {empathy_data.median():.0f}")
        print(f"  Range: {empathy_data.min():.0f} - {empathy_data.max():.0f}")
    
    if 'attachment_data' in locals() and len(attachment_data) > 0:
        print(f"\n📈 Attachment Scores (Outcome):")
        print(f"  Mean: {attachment_data.mean():.2f} (SD: {attachment_data.std():.2f})")
        print(f"  Median: {attachment_data.median():.0f}")
        print(f"  Range: {attachment_data.min():.0f} - {attachment_data.max():.0f}")
    
    if 'corr_coef' in locals():
        print(f"\n🔗 Correlation:")
        print(f"  Empathy ↔ Attachment: r = {corr_coef:.3f}")
    
    print(f"\n🎯 Data Quality:")
    missing_emp_pct = df_scored['empathy_score'].isna().sum() / len(df_scored) * 100
    missing_att_pct = df_scored['attachment_score'].isna().sum() / len(df_scored) * 100
    print(f"  Missing empathy: {missing_emp_pct:.2f}%")
    print(f"  Missing attachment: {missing_att_pct:.2f}%")
    
    remaining = TOTAL_CONVERSATIONS - len(df_scored)
    if remaining > 0:
        print(f"\n⏱️  Estimated time remaining:")
        print(f"  {remaining/(0.8*3600):.1f} hours at 0.8 conv/s")
    else:
        print(f"\n🎉 Scoring complete!")
    
    print("="*70)
else:
    print("⚠️  No data loaded. Run the cells above to load and analyze data.")

---

## 💡 Usage Tips

**How to use this notebook:**

1. **Run all cells** periodically (every few hours) to check progress
2. **Monitor the checkpoint file** as it grows
3. **Watch for data quality issues** early (missing scores, out-of-range values)
4. **Observe preliminary patterns** in distributions and correlations

**When to run:**
- After first checkpoint (1,000 conversations) to validate setup
- Every ~10,000 conversations to check quality
- Periodically during the 50-hour scoring process
- After completion for final analysis

**What to watch for:**
- ✅ Missing scores < 1% (acceptable)
- ✅ Scores in valid range (1-7)
- ✅ Reasonable distributions (not all 1s or 7s)
- ✅ Correlation makes sense (positive but not perfect)

**For full causal analysis**, wait until scoring is complete, then use:
- `03_analysis.ipynb` for propensity score matching and causal inference